# GRU workflow tool-sequence experiment

This notebook loads the workflow connection TSV, preprocesses it into tool sequences, and trains a simple GRU model to predict the next tool.

In [1]:
from pathlib import Path
import pandas as pd

root = Path("/home/john/icog-labs/galaxy-agent-xp-II")

tsv_path = root / "agents" / "expriments" / "data" / "worflow-connection-20-04.tsv"
print("TSV exists:", tsv_path.exists(), "->", tsv_path)

df = pd.read_csv(tsv_path, sep="\t")
print(df.shape)
df.head()

TSV exists: True -> /home/john/icog-labs/galaxy-agent-xp-II/agents/expriments/data/worflow-connection-20-04.tsv
(746138, 11)


,3,2013-02-07 16:48:46.721866,5,Grep1,1.0.1,7,Remove beginning1,1.0.0,f,t,f.1
0,3,2013-02-07 16:48:46.721866,6,Cut1,1.0.1,8,addValue,1.0.0,f,t,f
1,3,2013-02-07 16:48:46.721866,7,Remove beginning1,1.0.0,9,Cut1,1.0.1,f,t,f
2,3,2013-02-07 16:48:46.721866,7,Remove beginning1,1.0.0,6,Cut1,1.0.1,f,t,f
3,3,2013-02-07 16:48:46.721866,8,addValue,1.0.0,11,Paste1,1.0.0,f,t,f
4,3,2013-02-07 16:48:46.721866,9,Cut1,1.0.1,11,Paste1,1.0.0,f,t,f


In [2]:
import numpy as np

# Reload without assuming header row
raw = pd.read_csv(tsv_path, sep="\t", header=None)
raw.head()

,0,1,2,3,4,5,6,7,8,9,10
0,3,2013-02-07 16:48:46.721866,5,Grep1,1.0.1,7,Remove beginning1,1.0.0,f,t,f
1,3,2013-02-07 16:48:46.721866,6,Cut1,1.0.1,8,addValue,1.0.0,f,t,f
2,3,2013-02-07 16:48:46.721866,7,Remove beginning1,1.0.0,9,Cut1,1.0.1,f,t,f
3,3,2013-02-07 16:48:46.721866,7,Remove beginning1,1.0.0,6,Cut1,1.0.1,f,t,f
4,3,2013-02-07 16:48:46.721866,8,addValue,1.0.0,11,Paste1,1.0.0,f,t,f


In [3]:
# Assign column names (inferred) and basic cleanup
cols = [
    "workflow_id",
    "created_at",
    "source_step_id",
    "source_tool",
    "source_tool_version",
    "target_step_id",
    "target_tool",
    "target_tool_version",
    "flag_a",
    "flag_b",
    "flag_c",
]

raw.columns = cols

# Type conversions
raw["workflow_id"] = pd.to_numeric(raw["workflow_id"], errors="coerce").astype("Int64")
raw["source_step_id"] = pd.to_numeric(raw["source_step_id"], errors="coerce").astype("Int64")
raw["target_step_id"] = pd.to_numeric(raw["target_step_id"], errors="coerce").astype("Int64")

for c in ["flag_a", "flag_b", "flag_c"]:
    raw[c] = raw[c].map({"t": True, "f": False})

raw = raw.dropna(subset=["workflow_id", "source_step_id", "target_step_id", "source_tool", "target_tool"])
raw.head()

,workflow_id,created_at,source_step_id,source_tool,source_tool_version,target_step_id,target_tool,target_tool_version,flag_a,flag_b,flag_c
0,3,2013-02-07 16:48:46.721866,5,Grep1,1.0.1,7,Remove beginning1,1.0.0,False,True,False
1,3,2013-02-07 16:48:46.721866,6,Cut1,1.0.1,8,addValue,1.0.0,False,True,False
2,3,2013-02-07 16:48:46.721866,7,Remove beginning1,1.0.0,9,Cut1,1.0.1,False,True,False
3,3,2013-02-07 16:48:46.721866,7,Remove beginning1,1.0.0,6,Cut1,1.0.1,False,True,False
4,3,2013-02-07 16:48:46.721866,8,addValue,1.0.0,11,Paste1,1.0.0,False,True,False


In [4]:
from collections import defaultdict, deque

# Build tool sequences per workflow using a simple topological sort

def topo_sequence_for_workflow(df_wf):
    # Map step_id -> tool name (prefer source tool, then target tool)
    step_to_tool = {}
    for _, row in df_wf.iterrows():
        step_to_tool[int(row.source_step_id)] = row.source_tool
        step_to_tool[int(row.target_step_id)] = row.target_tool

    # Build adjacency and indegree
    adj = defaultdict(list)
    indeg = defaultdict(int)
    nodes = set()

    for _, row in df_wf.iterrows():
        s = int(row.source_step_id)
        t = int(row.target_step_id)
        adj[s].append(t)
        indeg[t] += 1
        nodes.add(s)
        nodes.add(t)
        indeg.setdefault(s, 0)

    # Kahn's algorithm with deterministic ordering
    queue = deque(sorted([n for n in nodes if indeg[n] == 0]))
    topo = []
    while queue:
        n = queue.popleft()
        topo.append(n)
        for m in adj.get(n, []):
            indeg[m] -= 1
            if indeg[m] == 0:
                queue.append(m)
        queue = deque(sorted(queue))

    if len(topo) != len(nodes):
        # Cycle or disconnected: fallback to sorted step ids
        topo = sorted(nodes)

    return [step_to_tool.get(step_id, "<UNK>") for step_id in topo]

# Create sequences (sample to keep memory reasonable for the demo)
workflow_ids = raw["workflow_id"].dropna().unique()
print("Total workflows:", len(workflow_ids))

sample_size = 5000  # adjust for speed
sample_ids = set(workflow_ids[:sample_size])

sequences = []
for wf_id, df_wf in raw[raw["workflow_id"].isin(sample_ids)].groupby("workflow_id"):
    seq = topo_sequence_for_workflow(df_wf)
    if len(seq) >= 2:
        sequences.append(seq)

print("Sequences:", len(sequences))
print("Example sequence:", sequences[0][:10])

Total workflows: 17270
Sequences: 5000
Example sequence: ['Grep1', 'Remove beginning1', 'Cut1', 'addValue', 'Cut1', 'Paste1', 'addValue']


In [5]:
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# Build vocab
special_tokens = ["<PAD>", "<UNK>"]
all_tools = sorted({tool for seq in sequences for tool in seq})
itos = special_tokens + all_tools
stoi = {tok: i for i, tok in enumerate(itos)}

pad_idx = stoi["<PAD>"]
unk_idx = stoi["<UNK>"]

context_len = 5

# Build training pairs
X, y = [], []
for seq in sequences:
    idxs = [stoi.get(t, unk_idx) for t in seq]
    for i in range(1, len(idxs)):
        start = max(0, i - context_len)
        context = idxs[start:i]
        if len(context) < context_len:
            context = [pad_idx] * (context_len - len(context)) + context
        X.append(context)
        y.append(idxs[i])

# Subsample for speed
max_samples = 200000
if len(X) > max_samples:
    X = X[:max_samples]
    y = y[:max_samples]

X = torch.tensor(X, dtype=torch.long)
y = torch.tensor(y, dtype=torch.long)

class ToolSeqDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


dataset = ToolSeqDataset(X, y)
loader = DataLoader(dataset, batch_size=256, shuffle=True)

class GRUPredictor(nn.Module):
    def __init__(self, vocab_size, emb_dim=64, hidden_dim=128):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, emb_dim, padding_idx=pad_idx)
        self.gru = nn.GRU(emb_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        emb = self.embedding(x)
        out, _ = self.gru(emb)
        last = out[:, -1, :]
        return self.fc(last)

model = GRUPredictor(len(itos))
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

# Train a few epochs
model.train()
for epoch in range(3):
    total_loss = 0.0
    for batch_X, batch_y in loader:
        optimizer.zero_grad()
        logits = model(batch_X)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    avg_loss = total_loss / len(loader)
    print(f"Epoch {epoch+1} | loss: {avg_loss:.4f}")

Epoch 1 | loss: 4.2024
Epoch 2 | loss: 2.4115
Epoch 3 | loss: 1.8424


In [6]:
def predict_next_tools(context_tools, top_k=5):
    model.eval()
    ctx = [stoi.get(t, unk_idx) for t in context_tools][-context_len:]
    if len(ctx) < context_len:
        ctx = [pad_idx] * (context_len - len(ctx)) + ctx
    x = torch.tensor([ctx], dtype=torch.long)
    with torch.no_grad():
        logits = model(x)[0]
        probs = torch.softmax(logits, dim=0)
        topk = torch.topk(probs, k=top_k)
    return [(itos[i], float(p)) for i, p in zip(topk.indices.tolist(), topk.values.tolist())]

# Example prediction
example_ctx = sequences[0][:context_len]
print("Context:", example_ctx)
print("Predicted next tools:")
for tool, score in predict_next_tools(example_ctx, top_k=5):
    print(f"  {tool}: {score:.3f}")

Context: ['Grep1', 'Remove beginning1', 'Cut1', 'addValue', 'Cut1']
Predicted next tools:
  Cut1: 0.313
  toolshed.g2.bx.psu.edu/repos/devteam/add_value/addValue/1.0.0: 0.044
  addValue: 0.040
  Paste1: 0.037
  toolshed.g2.bx.psu.edu/repos/devteam/tabular_to_fasta/tab2fasta/1.1.0: 0.030
